In [ ]:
import torch

def phi_init(
    Vgs,
    phi_f, # Surface potential
    T,     # Temperature
    NA,    # Acceptor concentration
    eps_sic, # Permittivity of SiC
    Cox,   # Oxide capacitance
    Vfbs0, # Flat-band voltage at T=0K
    Dit_mid, # Interface trap density at mid-gap
    Dit_edge, # Interface trap density at band edges
    sigma_it, # Standard deviation of interface trap energy distribution
    Eg,   # Bandgap energy
    phi_Fermi, # Fermi potential
):
    # 1. Thermal Voltage(V) phi_t = k_B * T / q 
    q = 1.602e-19 # Electron Charge
    phi_t = 1.381e-23 * T / q
    T = torch.tensor(T, dtype=torch.float64)

    # 2. Constant Gamma. 
    #    NA: accpetor doping concentration, from Product manufacturer
    #    eps_sic: permittivity of SiC:https://www.ioffe.ru/SVA/NSM/Semicond/SiC/basic.html), or from Product manufacturer
    #    Cox: oxide capacitance, from Product manufacturer(Cox = eps_ox / tox, tox: oxide thickness)
    gamma = torch.sqrt(2 * eps_sic * 1.602e-19 * NA) / Cox
    #    From raw paper of this methodology: gamma = torch.sqrt((2 * eps_sic * 1.602e-19 * NA)/phi_t) / Cox
    eps_sic = torch.tensor(eps_sic, dtype=torch.float64)
    NA = torch.tensor(NA, dtype=torch.float64)
    Cox = torch.tensor(Cox, dtype=torch.float64)

    # 3. Effective gate potential: uG
    #    Quasi-Fermi potential(phi_f)'s difference at the semiconductor surface: uf
    uG = (Vgs - Vfbs0) / phi_t
    uf = phi_f / phi_t

    # 4. α: Midgap interface-trap correction(?, inferredfrom GPT)
    alpha = 1.0 + q * Dit_mid / Cox

    # 5. Regions of SiC MOSFET operation
    #    Accumulation: If uG <= 0
    phis_init = (-2.0 * phi_t * torch.log(1.0 - uG / gamma))

    #    Depletion: If uG > 0
    u_dep = (((-gamma + torch.sqrt(gamma**2 + 4.0 * alpha * uG))/ (2.0 * alpha))**2)
    u_itc = (q* Dit_edge* sigma_it* torch.exp((-Eg / 2.0 - phi_Fermi)/ sigma_it)/ (phi_t * Cox))
    u_it0 = (uf + sigma_it / phi_t * torch.log(gamma / u_itc))
    u_si0 = (uf + 2.0 * phi_Fermi / phi_t)

    #    Weak Inversion: if uit0 <= u_dep
    u_it = (uf + sigma_it / phi_t * torch.log((uG - alpha * u_it0 - gamma * torch.sqrt(u_it0) + gamma) / u_itc))

    #    Strong Inversion: if usi0 > u_dep
    u_si = u_si0 + torch.log(((uG - u_si0) / gamma)**2 - u_si0 + 1)

    # FINAL: phis_init extraction
    phis_init = phi_t * torch.minimum(u_dep, torch.minimum(u_it, u_si))

    return phis_init

Calculation of phis_init. Workflow: Constant definition -> Region sortings -> phis_init calculated.
Phis_init approximation method: M.Albrecht et al., An Iterative Surface Potential Algorithm Including Interface Traps for Compact Modeling of SiC-MOSFETs.

In [ ]:
import torch
import torch.nn as nn


class DeltaPhisPINN(nn.Module):
    """
    delta_phi_s = NN(Vgs, phi_f, T)
    phi_s = phi_init + delta_phi_s
    1 layer, 8 neurons, Tanh activation function
    """

    def __init__(self):
        super().__init__()


        self.net = nn.Sequential(

            # Input: Vgs, Vds
            nn.Linear(2, 8),

            # Activation function used in the paper
            nn.Tanh(),

            # Output: delta_phi_s_PINN
            nn.Linear(8, 1)
        )

    def forward(self, Vgs, Vds):
        Vgs = Vgs.reshape(-1, 1)
        Vds = Vds.reshape(-1, 1)
        
        # Combine Vgs and Vds into a 1D tensor concatenation for further training
        x = torch.cat(
            [Vgs, Vds],
            dim=1)

        delta_phis_PINN = self.net(x)

        return delta_phis_PINN



PINN modeling

In [ ]:
def phi_PINN(
    model,
    Vgs,
    Vds,
    T,
    NA,
    eps_sic,
    Cox,
    Vfbs0,
    Dit_mid,
    Dit_edge,
    sigma_it,
    Eg,
    phi_Fermi,
):

    # According to the paper:
    # at drain side, phi_f = Vds
    phi_f = Vds

    # Initial guess from physical model
    phis_init = phi_init(
        Vgs,
        phi_f,
        T,
        NA,
        eps_sic,
        Cox,
        Vfbs0,
        Dit_mid,
        Dit_edge,
        sigma_it,
        Eg,
        phi_Fermi,
    )

    phis_init = phis_init.reshape(-1, 1)

    # PINN correction
    delta_phis_PINN = model(Vgs,Vds)

    # Final surface potential
    phis_PINN = (phis_init + delta_phis_PINN)

    return (
        phis_PINN,
        phis_init,
        delta_phis_PINN
    )

Final PINN surface potential
phi_s_PINN = phi_s_init + delta_phi_s_PINN

In [ ]:
# Physics-informed loss

def physics_loss(
    model,
    Vgs,
    Vds,
    T,
    NA,
    eps_sic,
    Cox,
    Vfbs0,
    Dit_mid,
    Dit_edge,
    sigma_it,
    Eg,
    phi_Fermi,
):

    # PINN prediction
    phis_PINN, phis_init, delta_phis_PINN = phi_PINN(
        model,
        Vgs,
        Vds,
        T,
        NA,
        eps_sic,
        Cox,
        Vfbs0,
        Dit_mid,
        Dit_edge,
        sigma_it,
        Eg,
        phi_Fermi,
    )

    # Physical gamma
    q = 1.602e-19
    gamma = torch.sqrt(2 * eps_sic * q * NA) / Cox

    # phi_f = Vds at drain side
    phi_f = Vds.reshape(-1, 1)

    # Surface-potential equation residual
    residual = (Vgs.reshape(-1, 1) - Vfbs - phis_PINN - gamma * H)
    
    # Two Elements of Resdiual L_SPE
    H = 
    Vfbs = 




    # L_SPE
    loss_SPE = torch.mean(residual ** 2)

    return loss_SPE

Pseudocode:
    residual = (Vgs.reshape(-1, 1) - Vfbs - phis_PINN - gamma * H)
    H = Equation (4)
    Vfb = Equation (7)
    ongoing



In [ ]:
# GPT: Training with Adam Optimizer according to the paper

Model = model(Vgs, Vds)

optimizer = torch.optim.Adam(Model.parameters(), lr=1e-4)
optimizer.zero_grad()
Model.train()
loss = []

for ep in range(self.epochs):


    loss.backward()

    optimizer.step()

Training part(ongoing)